# Data proc

## Null handling

### COALESCE vs IFNULL

The main difference between the two is that `IFNULL` function takes two arguments and returns the first one if it's not `NULL` or the second if the first one is `NULL`.

`COALESCE` function can take two or more parameters and returns the first non-`NULL` parameter, or `NULL` if all parameters are null, for example:

```sql
-- Source - https://stackoverflow.com/a/18528590
-- Posted by Aleks G, modified by community. See post 'Timeline' for change history
-- Retrieved 2026-01-19, License - CC BY-SA 4.0

SELECT IFNULL('some value', 'some other value');
-> returns 'some value'

SELECT IFNULL(NULL,'some other value');
-> returns 'some other value'

SELECT COALESCE(NULL, 'some other value');
-> returns 'some other value' - equivalent of the IFNULL function

SELECT COALESCE(NULL, 'some value', 'some other value');
-> returns 'some value'

SELECT COALESCE(NULL, NULL, NULL, NULL, 'first non-null value');
-> returns 'first non-null value'
```

## Outliers handling

### Quantiles

**BigQuery**

```sql
-- Outlier handling based on 5th and 95th quantile
WITH t_temp AS (
	SELECT 'Carpenter' AS profession, 70 AS age UNION ALL
	SELECT 'Carpenter', 50 UNION ALL
  SELECT 'Carpenter', 65 UNION ALL
  SELECT 'Carpenter', 45 UNION ALL
  SELECT 'Programmer', 30 UNION ALL
  SELECT 'Programmer', 35 UNION ALL
  SELECT 'Programmer', 20 UNION ALL
  SELECT 'Programmer', 25 
),

t_percentile AS (
  SELECT 
    *,
    PERCENTILE_CONT(age, 0.95) OVER(PARTITION BY profession) AS percentile_95,
    PERCENTILE_CONT(age, 0.05) OVER(PARTITION BY profession) AS percentile_5
  FROM t_temp
)

SELECT
  * EXCEPT (percentile_95, percentile_5)
FROM t_percentile
WHERE
  age > percentile_5
  AND age < percentile_95
/* It just removes ages 70 and 45 from `Carpenter`
and 20 and 35 from `Programmer`
*/
```

However, the problem with using quantiles for removing outliers is that it will remove any values from top and bottom, even if they are actually very close: 
```sql
WITH t_temp AS (
	SELECT 'Carpenter' AS profession, 1 AS age UNION ALL
	SELECT 'Carpenter', 2 UNION ALL
  SELECT 'Carpenter', 3 UNION ALL
  SELECT 'Carpenter', 4 UNION ALL
  SELECT 'Programmer', 10 UNION ALL
  SELECT 'Programmer', 11 UNION ALL
  SELECT 'Programmer', 12 UNION ALL
  SELECT 'Programmer', 13 
),

t_percentile AS (
  SELECT 
    *,
    PERCENTILE_CONT(age, 0.95) OVER(PARTITION BY profession) AS percentile_95,
    PERCENTILE_CONT(age, 0.05) OVER(PARTITION BY profession) AS percentile_5
  FROM t_temp
)

SELECT
  * EXCEPT (percentile_95, percentile_5)
FROM t_percentile
WHERE
  age > percentile_5
  AND age < percentile_95
-- this will remove rows with ages 1 and 4 from `Carpenter`
-- and rows with ages 10 and 13 from `Programmer`
```

### Z-score

```sql
WITH data1 AS (
	SELECT 1 category, 'protein' nutrient, 3 value
	UNION ALL 
	SELECT 1, 'protein', 5
	UNION ALL 
	SELECT 1, 'protein', 19
	UNION ALL 
	SELECT 1, 'protein', 9
	UNION ALL 
	SELECT 1, 'protein', 100
	UNION ALL 
	SELECT 1, 'fat', 7
	UNION ALL
	SELECT 1, 'fat', 8
	UNION ALL
	SELECT 1, 'fat', 9
	UNION ALL
	SELECT 1, 'fat', 39
	UNION ALL
	SELECT 1, 'fat', 2
	UNION ALL
	SELECT 2, 'protein', 15
	UNION ALL
	SELECT 2, 'protein', 10
	UNION ALL
	SELECT 2, 'protein', 1000
	UNION ALL
	SELECT 2, 'fat', 14
	UNION ALL
	SELECT 2, 'fat', 19
), 

intermediate_calc AS (
	SELECT
		category,
		nutrient,
		value,
		AVG(value) OVER (PARTITION BY category, nutrient) AS mu,
		COUNT(value) OVER (PARTITION BY category, nutrient) AS n,
		(value - AVG(value) OVER (PARTITION BY category, nutrient))
			* (value - AVG(value) OVER (PARTITION BY category, nutrient))
			AS x_minus_mu_squared
	FROM data1
),

calculate_sigma AS (
	SELECT
		category,
		nutrient,
		value,
		mu,
		n, 
		SQRT(SUM(x_minus_mu_squared) OVER (PARTITION BY category, nutrient) / n) AS sigma
	FROM intermediate_calc
)

SELECT
	category,
	nutrient,
	(value - mu) / sigma AS zscore
FROM calculate_sigma
ORDER BY category ASC, nutrient ASC
```

# Null handling

## First non-null value in time series

In [ ]:
/*
Option 1: Window function
- works in BigQuery

I don't like this one - difficult to read
*/
WITH sample_data AS (
  SELECT * FROM UNNEST([
    STRUCT('id_1' AS _id, 'shoes > running' AS tag,  'lightweight running shoes' AS description, DATE '2024-01-01' AS rundate),
          ('id_1',        NULL,                      'updated description v2',                   DATE '2024-01-05'),
          ('id_1',        'SSPorts > running shoes', NULL,                                       DATE '2024-01-10'),
          ('id_1',        NULL,                      NULL,                                       DATE '2024-01-15'),

          ('id_2',        NULL,                      'basic t-shirt',                            DATE '2024-02-01'),
          ('id_2',        'clothiING_ > tops',       NULL,                                       DATE '2024-02-03'),
          ('id_2',        NULL,                      'premium cotton t-shirt',                   DATE '2024-02-10'),

          ('id_3',        NULL,                      NULL,                                       DATE '2024-03-01'),
          ('id_3',        'electronics > audio',     'wireless headphones',                      DATE '2024-03-05'),
          ('id_3',        NULL,                      NULL,                                       DATE '2024-03-10')
  ])
)

SELECT
  _id,
  LAST_VALUE(tag IGNORE NULLS)
    OVER (
      PARTITION BY _id
      ORDER BY rundate
      ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS tag,
  LAST_VALUE(description IGNORE NULLS)
    OVER (
      PARTITION BY _id
      ORDER BY rundate
      ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS description
FROM sample_data
QUALIFY ROW_NUMBER() OVER (PARTITION BY _id ORDER BY rundate DESC) = 1

-- Fila	_id	tag	description
-- 1	id_1	SSPorts > running shoes	updated description v2
-- 2	id_2	clothiING_ > tops	premium cotton t-shirt
-- 3	id_3	electronics > audio	wireless headphones

In [ ]:
/*
Option 2: ARRAY_AGG
- works in BigQuery

Pros:
- Very explicit and easy to understand
- Avoids window frame completely

Cons:
- Slightly heavier than window functions on large tables
- less flexible for chaining

> I PREFER THIS ONE
*/
WITH sample_data AS (
  SELECT * FROM UNNEST([
    STRUCT('id_1' AS _id, 'shoes > running' AS tag,  'lightweight running shoes' AS description, DATE '2024-01-01' AS rundate),
          ('id_1',        NULL,                      'updated description v2',                   DATE '2024-01-05'),
          ('id_1',        'SSPorts > running shoes', NULL,                                       DATE '2024-01-10'),
          ('id_1',        NULL,                      NULL,                                       DATE '2024-01-15'),

          ('id_2',        NULL,                      'basic t-shirt',                            DATE '2024-02-01'),
          ('id_2',        'clothiING_ > tops',       NULL,                                       DATE '2024-02-03'),
          ('id_2',        NULL,                      'premium cotton t-shirt',                   DATE '2024-02-10'),

          ('id_3',        NULL,                      NULL,                                       DATE '2024-03-01'),
          ('id_3',        'electronics > audio',     'wireless headphones',                      DATE '2024-03-05'),
          ('id_3',        NULL,                      NULL,                                       DATE '2024-03-10')
  ])
)

SELECT
  _id,
  ARRAY_AGG(LOWER(tag)  IGNORE NULLS ORDER BY rundate DESC LIMIT 1)[OFFSET(0)] AS tag,
  ARRAY_AGG(description IGNORE NULLS ORDER BY rundate DESC LIMIT 1)[OFFSET(0)] AS description
FROM sample_data
GROUP BY _id

-- Fila	_id tag	                   description
-- 1	id_1	ssports > running shoes	   updated description v2
-- 2	id_2	clothiing_ > tops	         premium cotton t-shirt
-- 3	id_3	electronics > audio	       wireless headphones